## 1.       ABSA

In [2]:
# --- BLOC 1 : extraction d'aspects sur plusieurs avis, sans reseau ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from aspect_sentiment.absa import extract_aspect_candidates

avis_test = [
    "The delivery was slow but the product quality is excellent",
    "Customer service was rude and the refund took forever",
    "Great packaging, arrived in perfect condition, fast shipping",
]

for avis in avis_test:
    aspects = extract_aspect_candidates(avis)
    print(f"Avis : {avis}")
    print(f"  Aspects extraits : {aspects}\n")

/home/ensai/Documents/Repositories_Git/sentiment-analysis-platform/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Avis : The delivery was slow but the product quality is excellent
  Aspects extraits : ['delivery', 'product quality']

Avis : Customer service was rude and the refund took forever
  Aspects extraits : ['Customer service', 'refund']

Avis : Great packaging, arrived in perfect condition, fast shipping
  Aspects extraits : ['Great packaging', 'condition', 'shipping']



In [ ]:
# --- BLOC 2 : jeu de donnees ABSA etiquete a la main (petit, pour ---
# --- verification -- un vrai entrainement utiliserait SemEval) ---
# Format : (texte, aspect, label) -- label : 0=negative, 1=neutral, 2=positive
donnees_absa = [
    ("The delivery was slow but the product quality is excellent", "delivery", 0),
    (
        "The delivery was slow but the product quality is excellent",
        "product quality",
        2,
    ),
    ("Customer service was rude and the refund took forever", "customer service", 0),
    ("Customer service was rude and the refund took forever", "refund", 0),
    ("Great packaging, arrived in perfect condition, fast shipping", "packaging", 2),
    ("Great packaging, arrived in perfect condition, fast shipping", "shipping", 2),
    ("The price is a bit high but the warranty is solid", "price", 0),
    ("The price is a bit high but the warranty is solid", "warranty", 2),
    ("Website checkout was confusing, though delivery was on time", "website", 0),
    ("Website checkout was confusing, though delivery was on time", "delivery", 2),
] * 10  # repete pour donner un peu plus de signal au fine-tuning

textes = [t for t, a, li in donnees_absa]
aspects_col = [a for t, a, li in donnees_absa]
labels_col = [li for t, a, li in donnees_absa]

# split simple train/eval (80/20)
n_train = int(len(donnees_absa) * 0.8)
train_textes, eval_textes = textes[:n_train], textes[n_train:]
train_aspects, eval_aspects = aspects_col[:n_train], aspects_col[n_train:]
train_labels, eval_labels = labels_col[:n_train], labels_col[n_train:]

print(f"Train : {len(train_textes)} exemples, Eval : {len(eval_textes)}")

Train : 80 exemples, Eval : 20


In [ ]:
# --- BLOC 3 : entrainement du modele ABSA (classification jointe) ---
from aspect_sentiment.absa import load_absa_classifier, train_absa_model

model, tokenizer = load_absa_classifier(num_labels=3)
print("Parametres du modele ABSA :", sum(p.numel() for p in model.parameters()))

trainer = train_absa_model(
    model,
    tokenizer,
    train_textes,
    train_aspects,
    train_labels,
    eval_textes,
    eval_aspects,
    eval_labels,
    epochs=5,
)

In [ ]:
# --- BLOC 4 : evaluation ---
from transformers_arch.fine_tuning import evaluate_fine_tuned_model

resultats = evaluate_fine_tuned_model(trainer)
print("\nResultats d'evaluation ABSA :", resultats)
# A verifier : accuracy et f1 (macro, corrige pour le multi-classe)

In [ ]:
# --- BLOC 5 : prediction complete sur un NOUVEL avis multi-aspects ---
from aspect_sentiment.absa import predict_aspect_sentiment

nouvel_avis = (
    "The delivery was fast this time but customer service still "
    "needs improvement, though the price is fair"
)
aspects_detectes = extract_aspect_candidates(nouvel_avis)
print(f"\nNouvel avis : {nouvel_avis}")
print(f"Aspects detectes automatiquement : {aspects_detectes}")

resultats_finaux = predict_aspect_sentiment(
    nouvel_avis, aspects_detectes, model, tokenizer
)
print("\n=== SORTIE FINALE ABSA (aspect -> sentiment) ===")
for aspect, sentiment in resultats_finaux.items():
    print(f"  {aspect:20} -> {sentiment}")